# EXAI-ResumeIntel: Ontology Engine Demo

**Author:** Mithin Sagar S ([@mithinsagar](https://github.com/mithinsagar))  
**Institution:** Vellore Institute of Technology (VIT)

This notebook demonstrates the hierarchical skill ontology and implicit skill inference (e.g. YOLOv8 -> Computer Vision -> Machine Learning).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from core.ontology import get_engine, SKILL_ONTOLOGY, ROLE_SKILL_WEIGHTS

engine = get_engine()
print(f'Ontology loaded: {len(SKILL_ONTOLOGY)} canonical skill nodes')
print(f'Total aliases:    {sum(len(v["aliases"]) for v in SKILL_ONTOLOGY.values())}')
print(f'Supported roles:  {len(ROLE_SKILL_WEIGHTS)}')

## 1. Basic skill extraction

In [ ]:
text = (
    'Experienced Machine Learning Engineer with 4 years building '
    'CV pipelines using PyTorch and YOLOv8. Deployed models on AWS SageMaker. '
    'Strong Python skills, worked with COCO datasets, implemented custom '
    'SHAP-based explainability.'
)

skills = engine.extract_skills(text)
for skill, conf in sorted(skills.items(), key=lambda x: -x[1]):
    print(f'  {skill:<32} {conf:.3f}')

## 2. Implicit skill inference

Text that never mentions "machine learning" but should still trigger it via parent propagation.

In [ ]:
cv_only = 'Built YOLOv8 detection with COCO dataset. Used NMS and mAP evaluation. Anchor box tuning.'

skills = engine.extract_skills(cv_only)
print('Text: "' + cv_only + '"\n')
print('Extracted:')
for skill, conf in sorted(skills.items(), key=lambda x: -x[1]):
    print(f'  {skill:<32} {conf:.3f}')

assert 'computer_vision' in skills
assert 'machine_learning' in skills
print('\n[OK] YOLO/COCO correctly propagated to computer_vision and machine_learning')

## 3. Role requirements

In [ ]:
roles_to_show = ['MACHINE LEARNING ENGINEER', 'FINANCE', 'HR', 'ADVOCATE']

for role in roles_to_show:
    reqs = engine.get_role_requirements(role)
    print(f'\n=== {role} ===')
    for skill, weight in sorted(reqs.items(), key=lambda x: -x[1]):
        print(f'  {skill:<32} {weight:.2f}')

## 4. Cross-domain comparison

In [ ]:
domains = {
    'ML Engineer': 'PyTorch YOLOv8 BERT MLOps Docker Kubernetes SHAP LIME Python',
    'Financial Analyst': 'DCF valuation Bloomberg equity research M&A Excel VBA GAAP IFRS',
    'HR Manager': 'Talent acquisition Workday HRIS payroll DEI FMLA labor law compliance',
    'Chef': 'French cuisine plating menu development sous chef Michelin catering',
}

for label, text in domains.items():
    skills = engine.extract_skills(text)
    top3 = sorted(skills.items(), key=lambda x: -x[1])[:3]
    print(f'{label:<20} -> ' + ', '.join(f'{s} ({c:.2f})' for s, c in top3))

---
**End of ontology demo.** Continue with `03_xai_walkthrough.ipynb` for the full XAI pipeline.